 # Module 5

## RAG the hard way
Prerequisites

In [ ]:
# Required imports
import boto3
import json
import PyPDF2
from pprint import pp
from opensearchpy import (
    AWSV4SignerAuth,
    OpenSearch,
    RequestsHttpConnection,
)
region = "us-east-1"

### Extract text from PDF file

In [ ]:
pdf_path = "input/AnyCompany_financial_10K.pdf"
with open(pdf_path, 'rb') as file:
    reader = PyPDF2.PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()

### Split text into overlapping chunks

In [ ]:
chunk_size=1000
overlap=200
chunks = []
start = 0
while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start = end - overlap

print(f"The total number of chunks is {len(chunks)}")

### Function to get embedding of text using Bedrock Titan model

In [ ]:
bedrock = boto3.client('bedrock-runtime', region_name=region)

def get_embedding(text):
    body = json.dumps({"inputText": text})
    
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=body
    )
    return json.loads(response['body'].read())['embedding']

Test the function with the first chunk

In [ ]:
print(f"The first chunk: {chunks[0]}")
embedding_of_first_chunk = get_embedding(chunks[0])
print(f"Size of the generated embedding: {len(embedding_of_first_chunk)}")
print("Embedding of first chunk:")
print(embedding_of_first_chunk)

### Index chunks with embeddings

#### Initialize OpenSearch Serverless client

In [ ]:
# NextGen VECTORSEARCH collection `rag` (OpenSearch console → Collections)
# Do not use collection `test` — that is type SEARCH and knn returns "Unsupported query type".
collection_endpoint = "https://cxssyq7wloytp2hy9jh0.aoss.us-east-1.on.aws"
index_name = "financial-documents"

# Create the SigV4 object for authentication with the credentials of boto3 to the 
#  service Amazon OpenSearch Serverless (aoss)
credentials = boto3.Session().get_credentials()
awsauth = AWSV4SignerAuth(credentials, region, 'aoss')

# Create the OpenSearch client to directly search in the index
opensearch_client = OpenSearch(
    hosts=[{'host': collection_endpoint.replace('https://', ''), 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    pool_maxsize = 20
)

#### Create the kNN index
Collection type must be **Vector search**. A NextGen **Search** collection will 403 on `knn_vector`. Data access policy must include the SageMaker execution role, not just the IAM user who clicked Express Create. Wait ~60s after a policy change.

In [ ]:
# Titan Embed v2 → 1024 dimensions. Create once; ignore if it already exists.
index_body = {
    "settings": {"index.knn": True},
    "mappings": {
        "properties": {
            "embedding": {
                "type": "knn_vector",
                "dimension": 1024,
                "space_type": "l2",
            },
            "text_chunk": {"type": "text"},
        }
    },
}

if not opensearch_client.indices.exists(index=index_name):
    print(opensearch_client.indices.create(index=index_name, body=index_body))
else:
    print(f"Index {index_name} already exists")

#### Test the indexing

In [ ]:
# Create the document that will be sent to OpenSearch
doc = {
    "embedding": embedding_of_first_chunk,
    "text_chunk": chunks[0]
}

# Add the document containing the text and embedding to the index
opensearch_client.index(
    index=index_name,
    body=doc
)

#### Embed and index each chunk

In [ ]:
# For each chunk
for chunk in chunks:
    # Generate the embedding
    embedding = get_embedding(chunk)

    # Create the document that will be sent to OpenSearch
    doc = {
        "embedding": embedding,
        "text_chunk": chunk
    }

    # Add the document containing the text and embedding to the index
    opensearch_client.index(
        index=index_name,
        body=doc
    )

### Retrieval (search)

#### Search for a single document

In [ ]:
# Define the query
query = "liquid cash"

# Get the embedding of the query which returns a vector of 1024 dimensions
query_embedding = get_embedding(query)

# Retrieve 1 document
k = 1

# Craft the search body
search_body = {
    "size": k,
    "query": {
        "knn": {
            "embedding": {
                "vector": query_embedding,
                "k": k
            }
        }
    }
}

# Search for the embedding of the query
response = opensearch_client.search(index=index_name, body=search_body)

# Display the answer
print(json.dumps(response, indent=2))

#### Define the function to search for similar documents

In [ ]:
def search(query, k=3):
    # Get the embedding of the query
    query_embedding = get_embedding(query)
    
    # Create the search query
    search_body = {
        "size": k,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": k
                }
            }
        }
    }

    # Search for the embedding in OpenSearch
    response = opensearch_client.search(index=index_name, body=search_body)
    
    # Loop through the results and extract the text_chunk 
    results = []
    for hit in response['hits']['hits']:
        results.append(hit['_source']['text_chunk'])
    
    return results

### Augmented (augment the prompt)

In [ ]:
original_prompt = "What investments have AnyCompany made?"

# Get the relevant documents (or chunks)
relevant_docs = search(original_prompt)

In [ ]:
# Create document blocks
document_blocks = []
for i, doc in enumerate(relevant_docs):
    document_blocks.append({
        "document": {
            "name": f"financial_doc_{i+1}",
            "format": "txt",
            "source": {
                "bytes": doc.encode('utf-8') # Need to convert the doc to bytes
            }
        }
    })

# Craft message for Converse API
messages = [
    {
        "role": "user",
        "content": document_blocks + [
            {
                "text": f"Based on the provided documents, please answer: {original_prompt}"
            }
        ]
    }
]

# Using pprint to print the content as it contains bytes
pp(messages, indent=2, width=120)

### Generation

In [ ]:
# Call Converse API
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=messages
)

print(json.dumps(response, indent=2))